In [1]:
### Import libraries, prepare the full-period dataset, and generate the main data splits
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split, KFold
from tqdm import tqdm, trange
import function
import os

### Create the output directory
os.makedirs("../results/work/1", exist_ok=True)

### Load yield, socioeconomic, and environmental source data
yield_data = pd.read_csv("../data/yield.csv", engine="python").values[:, 1:]
social_data = pd.read_csv("../data/social.csv", engine="python").values[:, 1:]
natural_dataset = pd.read_csv("../data/natural.csv", engine="python")
natural_data = function.natural_feature_builder(natural_dataset).values

### Convert inputs to tensors and reshape them to the required dimensions
yield_data = torch.from_numpy(yield_data.astype("float")).float()
social_data = torch.from_numpy(social_data.astype("float")).float().reshape(-1, 22, 9)
natural_data = torch.from_numpy(natural_data.astype("float")).float().reshape(22, 12, -1, 8)[:, :9].permute(2, 0, 1, 3).reshape(654, 22, -1)

### Apply the shared completeness rule and construct sample-level arrays
county_index = np.arange(654).reshape(-1, 1)
yield_data, social_data, natural_data, county_index = function.filter_(yield_data, social_data, natural_data, county_index)
yield_data, social_data, natural_data, county_index = function.flatten(yield_data, social_data, natural_data, county_index)

### Generate the held-out test set and five cross-validation folds
data_index = np.arange(yield_data.size(0))
train_valid_index, test_index = train_test_split(data_index, test_size=0.15)
n_splits, kf_indices = 5, []
kf = KFold(n_splits=n_splits, shuffle=True)
for train_index, valid_index in kf.split(train_valid_index):
    kf_indices.append((train_valid_index[train_index], train_valid_index[valid_index]))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb3 in position 397: invalid start byte

In [ ]:
### Train six models under both feature settings and export the full-period results
### Define model configurations and output columns
model_iter = ["dnn", "cnn", "rnn", "lst", "gru", "att"]
num_instance_identity, require_explanation, batch_size, learning_rate, max_epoch, patience, train_breaker = 128, False, 32, 0.0003, 1000, 20, True
accu_columns = ["r2_base", "mse_base", "mae_base", "r2_addi", "mse_addi", "mae_addi"]
inst_columns = ["county", "yield", "mse_base_init", "scores_base_init", "mse_base_pret", "scores_base_pret", "mse_base_fine", "scores_base_fine",
                "mse_addi_init", "scores_addi_init", "mse_addi_pret", "scores_addi_pret", "mse_addi_fine", "scores_addi_fine"]

### Train and evaluate each model across the five folds
for model in tqdm(model_iter):
    for i in trange(n_splits):
        train_index, valid_index = kf_indices[i][0], kf_indices[i][1]
        yield_train, social_train, natural_train = yield_data[train_index], social_data[train_index], natural_data[train_index]
        yield_valid, social_valid, natural_valid = yield_data[valid_index], social_data[valid_index], natural_data[valid_index]
        yield_test, social_test, natural_test = yield_data[test_index], social_data[test_index], natural_data[test_index]
        yield_train_scaled, yield_valid_scaled, yield_test_scaled = function.scaler(yield_train, yield_valid, yield_test)
        social_train_scaled, social_valid_scaled, social_test_scaled = function.scaler(social_train, social_valid, social_test)
        natural_train_scaled, natural_valid_scaled, natural_test_scaled = function.scaler(natural_train, natural_valid, natural_test)
        output_base_kf = function.predictor(model, "baseline", num_instance_identity, require_explanation,
                                            yield_train_scaled, social_train_scaled, natural_train_scaled,
                                            yield_valid_scaled, social_valid_scaled, natural_valid_scaled,
                                            batch_size, learning_rate, max_epoch, patience, train_breaker,
                                            yield_test_scaled, social_test_scaled, natural_test_scaled)
        output_addi_kf = function.predictor(model, "addition", num_instance_identity, require_explanation,
                                            yield_train_scaled, social_train_scaled, natural_train_scaled,
                                            yield_valid_scaled, social_valid_scaled, natural_valid_scaled,
                                            batch_size, learning_rate, max_epoch, patience, train_breaker,
                                            yield_test_scaled, social_test_scaled, natural_test_scaled,)
        train_accu = np.concatenate([np.array(output_base_kf[0]), np.array(output_addi_kf[0])], 1)
        test_accu = np.concatenate([np.array(output_base_kf[1]), np.array(output_addi_kf[1])], 1)
        train_inst = np.concatenate([np.concatenate(output_base_kf[2])[:, :, 0], np.concatenate(output_addi_kf[2])[:, :, 0]]).T
        test_inst = np.concatenate([np.concatenate(output_base_kf[3])[:, :, 0], np.concatenate(output_addi_kf[3])[:, :, 0]]).T
        for j in ["train", "test"]:
            exec(f"{j}_accu_{model}_kf{i+1}_csv = pd.DataFrame({j}_accu, index=['init','pret','fine'], columns=accu_columns)")
            exec(f"inst = np.concatenate([county_index[{j}_index,0].reshape(-1,1), yield_{j}, {j}_inst], 1)")
            exec(f"{j}_inst_{model}_kf{i+1}_csv = pd.DataFrame(inst, columns=inst_columns)")

### Aggregate results across folds
for i in model_iter:
    exec(f"train_accu_{i}, test_accu_{i}, test_inst_{i} = [], [], []")
for i in range(n_splits):
    for j in model_iter:
        exec(f"train_accu_{j}.append([train_accu_{j}_kf{i+1}_csv.values])")
        exec(f"test_accu_{j}.append([test_accu_{j}_kf{i+1}_csv.values])")
        exec(f"test_inst_{j}.append([test_inst_{j}_kf{i+1}_csv.values])")
for i in model_iter:
    for j in ["train", "test"]:
        exec(f"{j}_accu_{i}_csv = pd.DataFrame(np.concatenate({j}_accu_{i}).mean(0), index=['init','pret','fine'], columns=accu_columns)")
    exec(f"test_inst_{i}_csv = pd.DataFrame(np.concatenate(test_inst_{i}).mean(0), columns=inst_columns)")
index = [pd.DataFrame(test_index.reshape(-1, 1))]
for i in range(n_splits):
    index.append(pd.DataFrame(kf_indices[i][0]))
    index.append(pd.DataFrame(kf_indices[i][1]))
index_csv = pd.concat(index, axis=1)
index_csv.columns = ["test", "kf1_t", "kf1_v", "kf2_t", "kf2_v", "kf3_t", "kf3_v", "kf4_t", "kf4_v", "kf5_t", "kf5_v"]

### Export the processed results
index_csv.to_csv("../results/work/1/index.csv", index=False)
for i in model_iter:
    for j in ["train", "test"]:
        exec(f"{j}_accu_{i}_csv.to_csv('../results/work/1/{j}_accu_{i}.csv')")
        for k in range(n_splits):
            exec(f"{j}_accu_{i}_kf{k+1}_csv.to_csv('../results/work/1/{j}_accu_{i}_kf{k+1}.csv')")
            exec(f"{j}_inst_{i}_kf{k+1}_csv.to_csv('../results/work/1/{j}_inst_{i}_kf{k+1}.csv', index=False)")
    exec(f"test_inst_{i}_csv.to_csv('../results/work/1/test_inst_{i}.csv', index=False)")